# A03 — Análise de dados com Bedrock e Code Interpreter

![Arquitetura do exercício](assets/code-agent.png)

## O que você vai aprender

Você envia um CSV sintético para uma sessão AWS, solicita uma análise exploratória e faz uma pergunta específica. Bedrock gera o código; Code Interpreter o executa sobre o arquivo recebido. A sessão mantém os arquivos entre as perguntas.

## Pré-requisitos e sequência

Use uma conta AWS com sessão autorizada em `us-west-2`, Python 3.12 e kernel próprio para este apêndice.
As células instalam as versões validadas e criam os recursos durante a execução. Execute de cima para baixo, incluindo a limpeza.
Uma conta basta; as permissões variam conforme o evento. Estes exemplos são independentes das aulas Santander.

## Como interpretar o resultado

O CSV contém **299.130 linhas**. A pergunta final sobre pessoas chamadas Kimberly que preferem Crocodile deve resultar em **120**. A validação exige execução das duas análises pela ferramenta e confirma o número final.

## Origem e adaptações

Derivado do [notebook oficial](https://github.com/awslabs/agentcore-samples/blob/0d46da96306fa692c9887d0461252bf5aeeee6c3/06-workshops/05-AgentCore-tools/01-Agent-Core-code-interpreter/03-advanced-data-analysis-with-agent-using-code-interpreter/strands-agent-advanced-data-analysis-code-interpreter.ipynb), revisão `0d46da96306f`.
Foram adicionados explicações, validações e dependências explícitas; os caminhos dos arquivos foram organizados neste apêndice.
O Browser usa a cadeia padrão de credenciais e nomes exclusivos. O exemplo de cálculo declara Claude Haiku 4.5.
As evidências e diferenças constam no [relatório de validação](VALIDACAO.md).

## Conferir a sessão AWS


In [ ]:
import boto3, os
session = boto3.Session()
REGION = session.region_name or os.environ.get("AWS_DEFAULT_REGION")
assert REGION == "us-west-2", "Este apêndice foi validado em us-west-2; configure a região antes de continuar."
identity = session.client("sts").get_caller_identity()
print({"conta": identity["Account"], "regiao": REGION, "identidade": identity["Arn"]})
# Credenciais vêm da cadeia padrão do boto3: role do ambiente, perfil ou sessão temporária.
# Nenhuma chave, senha ou token deve ser salvo neste notebook.


In [ ]:
%pip install "bedrock-agentcore==1.25.3" "boto3==1.43.124" "strands-agents==1.64.0" "pandas==3.1.1"


In [ ]:
from bedrock_agentcore.tools.code_interpreter_client import CodeInterpreter
from strands import Agent, tool
from strands.models import BedrockModel
import json
import pandas as pd
from typing import Dict, Any

# Initialize the Code Interpreter within a supported AWS region.
code_client = CodeInterpreter("us-west-2")
code_client.start(session_timeout_seconds=1200)

## 2. Reading Local Data File

Now we'll read the contents of our sample data file. The file consists of random data with 4 columns: Name, Preferred_City, Preferred_Animal, Preferred_Thing and ~ 300,000 records.

We will analyze this file using an agent little later, to understand distributions and outliers

In [ ]:
df_data = pd.read_csv("assets/data.csv")
df_data.head()

In [ ]:
def read_file(file_path: str) -> str:
    """Helper function to read file content with error handling"""
    try:
        with open(file_path, "r", encoding="utf-8") as file:
            return file.read()
    except FileNotFoundError:
        print(f"Error: The file '{file_path}' was not found.")
        return ""
    except Exception as e:
        print(f"An error occurred: {e}")
        return ""


data_file_content = read_file("assets/data.csv")

## 3. Preparing Files for Sandbox Environment

We'll create a structure that defines the files we want to create in the sandbox environment.

In [ ]:
files_to_create = [{"path": "data.csv", "text": data_file_content}]

## 4. Creating Helper Function for Tool Invocation

This helper function will make it easier to call sandbox tools and handle their responses. Within an active session, you can execute code in supported languages (Python, JavaScript), access libraries based on your dependencies configuration, generate visualizations, and maintain state between executions.

In [ ]:
def call_tool(tool_name: str, arguments: Dict[str, Any]) -> Dict[str, Any]:
    """Helper function to invoke sandbox tools

    Args:
        tool_name (str): Name of the tool to invoke
        arguments (Dict[str, Any]): Arguments to pass to the tool

    Returns:
        Dict[str, Any]: JSON formatted result
    """
    response = code_client.invoke(tool_name, arguments)
    for event in response["stream"]:
        return json.dumps(event["result"])

## 5. Write data file to Code Sandbox

Now we'll write our data file into the sandbox environment and verify they were created successfully.

In [ ]:
# Write files to sandbox
writing_files = call_tool("writeFiles", {"content": files_to_create})
print("Writing files result:")
print(writing_files)

# Verify files were created
listing_files = call_tool("listFiles", {"path": ""})
print("\nFiles in sandbox:")
print(listing_files)

## 6. Perform Advanced Analysis using Strands based Agent

Now we will configure an agent to perform data analysis on the data file that we uploaded into the sandbox(above)

### 6.1 System Prompt Definition
Define the behavior and capabilities of the AI assistant. We instruct our assistant to always validate answers through code execution and data based reasoning.

In [ ]:
SYSTEM_PROMPT = """You are a helpful AI assistant that validates all answers through code execution using the tools provided. DO NOT Answer questions without using the tools

VALIDATION PRINCIPLES:
1. When making claims about code, algorithms, or calculations - write code to verify them
2. Use execute_python to test mathematical calculations, algorithms, and logic
3. Create test scripts to validate your understanding before giving answers
4. Always show your work with actual code execution
5. If uncertain, explicitly state limitations and validate what you can

APPROACH:
- If asked about a programming concept, implement it in code to demonstrate
- If asked for calculations, compute them programmatically AND show the code
- If implementing algorithms, include test cases to prove correctness
- Document your validation process for transparency
- The sandbox maintains state between executions, so you can refer to previous results

TOOL AVAILABLE:
- execute_python: Run Python code and see output

RESPONSE FORMAT: The execute_python tool returns a JSON response with:
- sessionId: The sandbox session ID
- id: Request ID
- isError: Boolean indicating if there was an error
- content: Array of content objects with type and text/data
- structuredContent: For code execution, includes stdout, stderr, exitCode, executionTime

For successful code execution, the output will be in content[0].text and also in structuredContent.stdout.
Check isError field to see if there was an error.

Be thorough, accurate, and always validate your answers when possible."""

### 6.2 Code Execution Tool Definition
Next we define the function as tool that will be used by the Agent as tool, to run code in the code sandbox. We use the @tool decorator to annotate the function as a custom tool for the Agent.

Within an active code interpreter session, you can execute code in supported languages (Python, JavaScript), access libraries based on your dependencies configuration, generate visualizations, and maintain state between executions.

In [ ]:
# Define and configure the code interpreter tool
@tool
def execute_python(code: str, description: str = "") -> str:
    """Execute Python code in the sandbox."""

    if description:
        code = f"# {description}\n{code}"

    # Print generated Code to be executed
    print(f"\n Generated Code: {code}")

    # Call the Invoke method and execute the generated code, within the initialized code interpreter session
    response = code_client.invoke("executeCode", {"code": code, "language": "python", "clearContext": False})
    for event in response["stream"]:
        return json.dumps(event["result"])

### 6.3 Agent Configuration
We create and configure an agent using the Strands SDK. We provide it the system prompt and the tool we defined above to execute generate code.

We use the Claude Haiku 4.5 model and we specify the [Cross-region Inference (CRIS)](https://docs.aws.amazon.com/bedrock/latest/userguide/cross-region-inference.html) profile id

In [ ]:
model_id = "global.anthropic.claude-haiku-4-5-20251001-v1:0"
model = BedrockModel(model_id=model_id)

# configure the strands agent including the model and tool(s)
agent = Agent(
    model=model,
    tools=[execute_python],
    system_prompt=SYSTEM_PROMPT,
    callback_handler=None,
)

## 7. Agent Invocation and Response Processing
We invoke the agent with our query and process the agent's response


Note: Async execution requires running in an async environment

## 7.1 Query to perform Exploratory Data Analysis(EDA)

Let's start with a query which instructs the agent to perform exploratory data analysis on the data file in the code sandbox environment

In [ ]:
query = "Load the file 'data.csv' and perform exploratory data analysis(EDA) on it. Tell me about distributions and outlier values."

# Invoke the agent asynchcronously and stream the response
response_text = ""
async for event in agent.stream_async(query):
    if "data" in event:
        # Stream text response
        chunk = event["data"]
        response_text += chunk
        print(chunk, end="")

## 7.2 Query to extract information

Now, let's instruct the agent to extract specific information from the data file in the code sandbox environment

In [ ]:
query = "Within the file 'data.csv', how many individuals with the first name 'Kimberly' have 'Crocodile' as their favourite animal?"

# Invoke the agent asynchcronously and stream the response
response_text = ""
async for event in agent.stream_async(query):
    if "data" in event:
        # Stream text response
        chunk = event["data"]
        response_text += chunk
        print(chunk, end="")

## 8. Cleanup

Finally, we'll clean up by stopping the Code Interpreter session. Once finished using a session, the session should be shopped to release resources and avoid unnecessary charges.

## Validar o resultado observado

As verificações abaixo falham se o resultado real não corresponder ao exercício.

In [ ]:
assert json.loads(writing_files)["isError"] is False
assert json.loads(listing_files)["isError"] is False
assert "data.csv" in listing_files
assert response_text.strip(), "No final answer"
assert "error" not in response_text.lower() and "unable" not in response_text.lower(), response_text
tool_results = [
    block["toolResult"]
    for message in agent.messages
    for block in message.get("content", [])
    if "toolResult" in block
]
assert len(tool_results) >= 2, "Expected AWS code execution for both questions"
assert all(r.get("status") != "error" for r in tool_results)
for result in tool_results:
    for block in result.get("content", []):
        if "text" in block:
            payload = json.loads(block["text"])
            assert payload.get("isError") is False, payload
print("APPENDIX_RESULT_VALIDATED: both analysis questions executed using AWS Code Interpreter")
import re
assert re.search(r"\b120\b", response_text), "Esperado: 120 pessoas Kimberly / Crocodile."


In [ ]:
# Stop the Code Interpreter session
code_client.stop()
print("Code Interpreter session stopped successfully!")